# FinRAG — asking questions of SEC filings

**The Gen Academy · Week 2 · Project 2**

Answers analyst questions about eight semiconductor companies from their latest
annual reports (10-K filings), with citations — and refuses when the filings
don't contain the answer.

Runs locally in VS Code. Click the **▶** to the left of a cell to run it, then
move to the next. Read the output of each one before moving on — that reading
is most of the learning.

**Before cell 1:** make sure your virtual environment is active and `.env`
holds your two keys. See `SETUP.md`.

## 1 · Load your keys

Your API keys live in a file called `.env` that Git is told to ignore, so they
never reach your public repo. `load_dotenv()` reads that file and puts the
keys into this program's environment.

This prints only whether each key was *found* — never the key itself. Get in
the habit: keys don't get printed, ever.

In [8]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

from dotenv import load_dotenv
import os

load_dotenv()

for name in ('NEBIUS_API_KEY', 'PINECONE_API_KEY'):
    key = os.getenv(name)
    print(f'{name:20s} {"found ("+str(len(key))+" characters)" if key else "MISSING"}')

project root: /Users/vj/Desktop/Mastering Agentic AI - The Gen Academy/Projects/Week 2/FinRAG
NEBIUS_API_KEY       found (235 characters)
PINECONE_API_KEY     found (75 characters)


> **If either says MISSING:** the `.env` file isn't where Python is looking,
> or a name is misspelled. It must sit in the project's top folder, and the
> names must match exactly — capitals and underscores included.

## 2 · See which models your account has

Two models do two very different jobs:

- the **embedding model** turns text into numbers, so passages can be *found*
- the **chat model** reads the passages that were found and *writes* the answer

Rather than guessing names, this asks Nebius what your account can actually
use. Copy two of them into the next cell.

In [9]:
from openai import OpenAI

BASE_URL = 'https://api.tokenfactory.nebius.com/v1/'
probe = OpenAI(base_url=BASE_URL, api_key=os.environ['NEBIUS_API_KEY'])
models = sorted(m.id for m in probe.models.list().data)

print(f'{len(models)} models available on your account\n')
print('EMBEDDING models — pick one for EMBED_MODEL:')
for m in models:
    if any(k in m.lower() for k in ('embed', 'bge', 'e5')):
        print('   ', m)

print('\nCHAT models — pick one for CHAT_MODEL:')
for m in models:
    if not any(k in m.lower() for k in ('embed', 'bge', 'e5')):
        print('   ', m)

30 models available on your account

EMBEDDING models — pick one for EMBED_MODEL:
    Qwen/Qwen3-Embedding-8B

CHAT models — pick one for CHAT_MODEL:
    MiniMaxAI/MiniMax-M2.5
    MiniMaxAI/MiniMax-M3
    NousResearch/Hermes-4-405B
    NousResearch/Hermes-4-70B
    Qwen/Qwen2.5-VL-72B-Instruct
    Qwen/Qwen3-235B-A22B-Instruct-2507
    Qwen/Qwen3-30B-A3B-Instruct-2507
    Qwen/Qwen3-32B
    Qwen/Qwen3-Next-80B-A3B-Thinking
    Qwen/Qwen3.5-397B-A17B
    deepseek-ai/DeepSeek-V4-Flash
    deepseek-ai/DeepSeek-V4-Flash-0731
    deepseek-ai/DeepSeek-V4-Pro
    google/gemma-3-27b-it
    meta-llama/Llama-3.3-70B-Instruct
    moonshotai/Kimi-K2.6
    moonshotai/Kimi-K2.7-Code
    moonshotai/Kimi-K3
    nvidia/Cosmos3-Super-Reasoner
    nvidia/Llama-3_1-Nemotron-Ultra-253B-v1
    nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B
    nvidia/Nemotron-3-Nano-Omni
    nvidia/Nemotron-3-Ultra-550b-a55b
    nvidia/Nemotron-3_5-Lightning
    nvidia/nemotron-3-super-120b-a12b
    openai/gpt-oss-120b
    openbmb/

## 3 · Choose your two models and test them

Paste two names from the list above. The test does one of each job, so if
something is wrong with your key or a model name you find out here — before
anything expensive runs.

Note what it prints for **dimension**. That's how many numbers the embedding
model produces per passage, and Pinecone needs to be told the same figure.

In [10]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

EMBED_MODEL = 'Qwen/Qwen3-Embedding-8B'
CHAT_MODEL  = 'Qwen/Qwen3-235B-A22B-Instruct-2507'     # <- replace with one from the list

embeddings = OpenAIEmbeddings(model=EMBED_MODEL, base_url=BASE_URL,
                              api_key=os.environ['NEBIUS_API_KEY'],
                              check_embedding_ctx_length=False)
llm = ChatOpenAI(model=CHAT_MODEL, base_url=BASE_URL,
                 api_key=os.environ['NEBIUS_API_KEY'], temperature=0)

EMBED_DIM = len(embeddings.embed_query('Data Center revenue 47,525'))
print(f'embedding model OK  ->  dimension = {EMBED_DIM}')
print('chat model OK       -> ', llm.invoke('Reply with exactly: ready.').content.strip()[:40])
print(f'\nRemember {EMBED_DIM}. Pinecone will need it.')

embedding model OK  ->  dimension = 4096
chat model OK       ->  ready.

Remember 4096. Pinecone will need it.


## 4 · Download the filings

**You don't download these by hand.** This cell does it.

SEC EDGAR is the US government's public archive of company filings. Every
public company must file a 10-K each year — a long, factual annual report.

Two lookups happen here, and the first one is worth understanding:

**Ticker → CIK.** EDGAR doesn't know what "NVDA" means. It identifies companies
by **CIK**, a permanent number the SEC assigns that never changes even if a
company renames or re-lists. Nobody knows CIKs by heart, and a wrong one fails
badly — you'd silently get somebody else's filings rather than an error. So the
code reads the SEC's own published ticker→CIK mapping and resolves them at
runtime. That's also exactly the lookup an "add any company" feature would need,
so it's already half-built.

**CIK → newest 10-K.** Ask EDGAR for that company's filing history, keep only
entries of type `10-K`, take the most recent, download it.

Doing all this in code rather than by hand is deliberate: it's repeatable, it
always fetches whatever is *currently* newest, and anyone cloning your repo
gets the same corpus without you emailing them anything.

The SEC asks API users to identify themselves in each request — the `UA` line
in `src/fetch_filings.py` does that. It's a courtesy requirement of their
fair-access policy, and it's already set.

Eight companies, all semiconductors. One sector matters: it means comparison
questions are meaningful rather than apples-to-oranges.

In [11]:
from src.fetch_filings import fetch_corpus
import pathlib

RAW = pathlib.Path('data/raw')
manifest = fetch_corpus(RAW)

resolving 8 tickers to CIKs ...
  NVIDIA               CIK 1045810
  AMD                  CIK 2488
  Intel                CIK 50863
  Broadcom             CIK 1730168
  Qualcomm             CIK 804328
  Micron               CIK 723125
  Texas Instruments    CIK 97476
  Applied Materials    CIK 6951

  cached NVIDIA               2026-02-25  1.97 MB
  cached AMD                  2026-02-04  2.17 MB
  cached Intel                2026-01-23  3.32 MB
  cached Broadcom             2025-12-18  2.70 MB
  cached Qualcomm             2025-11-05  1.88 MB
  cached Micron               2025-10-03  2.44 MB
  cached Texas Instruments    2026-02-06  2.11 MB
  cached Applied Materials    2025-12-12  2.22 MB

8/8 filings ready


Look in `data/raw/` in VS Code's file explorer — eight `.html` files, a
couple of megabytes each. Open one if you're curious: it's an unreadable wall
of markup. Making it readable is the next step.

They're cached, so re-running that cell costs nothing. `data/` is gitignored,
so these don't bloat your repo.

## 5 · Clean the HTML

**Read this output carefully.** Real filings are far messier than any test
file, and problems are cheap to catch here and expensive to discover later.

A generic HTML reader breaks on a 10-K in two specific ways:

- Filings use `<table>` for real financial data **and** for page layout. Flatten
  both and `Data Center 115,186 47,525` arrives with no clue which number
  belongs to which year.
- EDGAR puts the `$` in its own cell, so a row comes through as
  `['Data Center', '$', '115,186', '$', '47,525']` against a 4-column header.
  Left alone, the columns stop lining up entirely.

`src/clean.py` sorts real tables from layout ones, folds the currency cells
back into their values, and tags every block with the Item section it came
from — which is what lets an answer cite *"NVIDIA, Item 7"* instead of a
meaningless chunk number.

Check: Items 1, 1A, 7 and 8 present for each company, and *hundreds* of
tables rather than a handful.

In [18]:
from src.clean import html_to_blocks, corpus_stats
from src.to_documents import blocks_to_documents, describe

all_blocks = []
for m in manifest:
    blocks = html_to_blocks(pathlib.Path(m['path']).read_bytes(),
                            company=m['company'], filing_date=m['filing_date'],
                            source_url=m['url'], scheme='item')
    st = corpus_stats(blocks)
    print(f"{m['company']:9s} blocks={st['blocks']:5d} tables={st['table_blocks']:4d} "
          f"chars={st['chars']/1000:7.0f}k sections={len(st['sections'])}")
    all_blocks.extend(blocks)

DOCS = blocks_to_documents(all_blocks)
print(f'\nTOTAL {len(DOCS)} documents ready for LangChain')

NVIDIA    blocks=  763 tables=  58 chars=    352k sections=20
AMD       blocks=  743 tables=  62 chars=    433k sections=18
Intel     blocks= 1030 tables=  89 chars=    575k sections=13
Broadcom  blocks=  841 tables=  82 chars=    409k sections=18
Qualcomm  blocks=  697 tables=  57 chars=    410k sections=17
Micron    blocks=  739 tables=  66 chars=    379k sections=17
Texas Instruments blocks=  519 tables=  70 chars=    222k sections=16
Applied Materials blocks=  663 tables=  64 chars=    319k sections=17

TOTAL 5995 documents ready for LangChain


In [13]:
# Look at a real extracted table. Do the numbers line up under their headers?
tables = [d for d in DOCS if d.metadata['is_table'] and 150 < len(d.page_content) < 900]
print(describe(tables, n=2))

--- NVIDIA · Front Matter · TABLE · 240 chars
Delaware | 94-3177549
(State or other jurisdiction of | (I.R.S. Employer
incorporation or organization) | Identification No.)
2788 San Tomas Expressway , Santa Clara , California | 95051
(Address of principal executive offices) | (Zip Code)

--- NVIDIA · Item 1 - Business · TABLE · 360 chars
Name | Age | Position
Jen-Hsun Huang | 63 | President and Chief Executive Officer
Colette M. Kress | 58 | Executive Vice President and Chief Financial Officer
Ajay K. Puri | 71 | Executive Vice President, Worldwide Field Operations
Debora Shoquist | 71 | Executive Vice President, Operations
Timothy S. Teter | 59 | Executive Vice President and General Counsel



## 6 · Chunk it — two ways · **graded comparison #1**

A whole filing won't fit in a model's context window, and you wouldn't want it
to — more text means more to get confused by. So documents get cut into small
passages called chunks.

Two ways to cut, and comparing them is one of your two requirements:

- **Fixed** — cut every ~800 characters, with a 100-character overlap so a
  sentence split across the boundary still appears whole somewhere. Completely
  blind to what the text says. This is `RecursiveCharacterTextSplitter`, the
  one Session 1 recommended starting with.
- **Semantic** — cut where the *topic* changes, measured by comparing the
  meaning of consecutive sentences. Chunk sizes vary, because topics do.

One rule applies to both: **tables are never split.** If only one strategy
protected tables, you couldn't tell whether a difference came from where the
boundaries fall or from how tables were treated. Holding it constant is what
makes the comparison mean anything — and that reasoning is worth saying in
your write-up.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from collections import defaultdict
import time

prose  = [d for d in DOCS if not d.metadata['is_table']]
tables = [d for d in DOCS if d.metadata['is_table']]

# Join the paragraphs back into one document per (company, section).
#
# The cleaner deliberately emits small blocks so tables can be isolated and
# every block tagged with its Item section. But a chunker needs CONTINUOUS
# text: semantic chunking looks for the point where a topic shifts, and a lone
# paragraph has no such point. Handing it 5,000 paragraphs would mean 5,000
# separate round trips to the embedding model, each one asking a question with
# no meaningful answer.
#
# Merging also keeps the comparison honest. Both strategies now receive
# IDENTICAL input, so the only thing that differs between them is where they
# place boundaries -- which is exactly what the comparison is meant to isolate.
buckets = defaultdict(list)
for d in prose:
    buckets[(d.metadata['company'], d.metadata['section'])].append(d)

SECTIONS = []
for (company, section), docs in buckets.items():
    docs.sort(key=lambda d: d.metadata['order'])
    SECTIONS.append(Document(
        page_content='\n\n'.join(d.page_content for d in docs),
        metadata=dict(docs[0].metadata)))

print(f'{len(prose)} prose blocks -> {len(SECTIONS)} section documents')
print(f'{len(tables)} tables (kept whole by both strategies)\n')

fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
FIXED = fixed_splitter.split_documents(SECTIONS) + tables
print(f'fixed: {len(FIXED)} chunks')

5447 prose blocks -> 136 section documents
548 tables (kept whole by both strategies)

fixed: 5451 chunks


The next cell is the slow one — semantic chunking sends every sentence to the
embedding model to work out where the topic shifts. It will sit there for a
while. That's it working, not hanging.

In [15]:
from langchain_experimental.text_splitter import SemanticChunker

t0 = time.time()
semantic_splitter = SemanticChunker(embeddings,
                                    breakpoint_threshold_type='percentile',
                                    breakpoint_threshold_amount=90)
SEMANTIC = semantic_splitter.split_documents(SECTIONS) + tables
print(f'semantic: {len(SEMANTIC)} chunks  ({time.time()-t0:.0f}s)')

/var/folders/zc/gzc08x0938z8ftdp_s5pkjbr0000gn/T/ipykernel_76764/735624588.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


semantic: 2107 chunks  (374s)


In [16]:
import statistics as st
for name, chunks in (('fixed', FIXED), ('semantic', SEMANTIC)):
    lens = [len(c.page_content) for c in chunks]
    print(f'{name:9s} n={len(chunks):5d}  chars: mean={st.mean(lens):6.0f} '
          f'median={st.median(lens):6.0f} min={min(lens):5d} max={max(lens):6d}')

print('\nFixed chunks are all about the same size — that is what "fixed" means.')
print('Semantic sizes vary, because topics vary in length.')

fixed     n= 5451  chars: mean=   595 median=   645 min=   57 max=  5379
semantic  n= 2107  chars: mean=  1474 median=   768 min=    2 max= 18584

Fixed chunks are all about the same size — that is what "fixed" means.
Semantic sizes vary, because topics vary in length.


In [19]:
MIN_CHARS, MAX_CHARS = 50, 2500

def bound(chunks):
    """
    Semantic chunking finds topic boundaries but guarantees nothing about size.
    Left alone it produces both 2-character slivers (no information) and
    18,000-character monsters (flood the context, and the reranker only reads
    the first ~512 tokens anyway). So: drop the slivers, split the monsters.

    Tables are exempt -- a table split in half is useless, which is the whole
    reason both strategies keep them whole.
    """
    hard = RecursiveCharacterTextSplitter(chunk_size=MAX_CHARS, chunk_overlap=100)
    out = []
    for c in chunks:
        if c.metadata.get('is_table'):
            out.append(c)
        elif len(c.page_content) < MIN_CHARS:
            pass
        elif len(c.page_content) > MAX_CHARS:
            out.extend(hard.split_documents([c]))
        else:
            out.append(c)
    return out

before = len(SEMANTIC)
SEMANTIC = bound(SEMANTIC)
FIXED    = bound(FIXED)          # no-op, but applied to both so it stays fair
print(f'semantic: {before} -> {len(SEMANTIC)} chunks after bounding')
print(f'fixed:    {len(FIXED)} chunks')

semantic: 2693 -> 2692 chunks after bounding
fixed:    5451 chunks


## 7 · Put them in Pinecone

Pinecone holds all those number-lists and finds the nearest ones instantly.

Two things to understand here:

**Dimension.** An index is created with a fixed number of dimensions and it
can never change. It must match your embedding model exactly — the `EMBED_DIM`
printed back in cell 3. Mismatch it and every insert fails. This cell reads the
number from the model rather than hard-coding it, which is why that can't go
wrong here.

**Namespaces.** Rather than two separate indexes for the two chunking
strategies, we use one index with two *namespaces* — separate compartments
inside it. Searching one namespace never sees the other, so the comparison
stays clean, and you stay inside the free plan's index limit.

In [20]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = 'finrag'

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

existing = [i['name'] for i in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(name=INDEX_NAME, dimension=EMBED_DIM, metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region='us-east-1'))
    print(f'created index "{INDEX_NAME}" with dimension {EMBED_DIM}')
else:
    got = pc.describe_index(INDEX_NAME)['dimension']
    print(f'index "{INDEX_NAME}" already exists, dimension {got}')
    if got != EMBED_DIM:
        raise SystemExit(
            f'MISMATCH: index is {got} but your embedding model makes {EMBED_DIM}.\n'
            f'Delete the index in the Pinecone console and re-run this cell.')

created index "finrag" with dimension 4096


This cell uploads every chunk. It's the slow, expensive step — each chunk goes
to the embedding model and then to Pinecone. It runs once.

While it runs, open **app.pinecone.io** and watch the vector count climb.

In [21]:
STORES = {}
for ns, chunks in (('fixed', FIXED), ('semantic', SEMANTIC)):
    t0 = time.time()
    STORES[ns] = PineconeVectorStore.from_documents(
        chunks, embeddings, index_name=INDEX_NAME, namespace=ns)
    print(f'{ns:9s} uploaded {len(chunks):5d} chunks in {time.time()-t0:.0f}s')

print('\nOpen app.pinecone.io -> your "finrag" index. Click a vector to see')
print('the numbers, the original text, and the metadata travelling with it.')

fixed     uploaded  5451 chunks in 158s
semantic  uploaded  2692 chunks in 110s

Open app.pinecone.io -> your "finrag" index. Click a vector to see
the numbers, the original text, and the metadata travelling with it.


## 8 · Retrieval — and why we use two kinds

Two ways to search, failing in opposite directions:

- **Vector search** finds passages that *mean* something similar. Excellent for
  "why did revenue change". Weak when the answer hinges on an exact string.
- **BM25** is keyword matching — a smarter Ctrl-F. Can't paraphrase at all, but
  never fumbles a literal term.

Financial questions live on exact terms: segment names, fiscal years, figures.
So we run both and merge them. `EnsembleRetriever` does the merging using
*reciprocal rank fusion* — whatever ranks high in **both** lists wins.

Run the next cell and watch vector search drift while BM25 goes straight to
`Data Center`. That contrast is the whole argument for hybrid, and it's worth
showing in your video.

In [22]:
from langchain_community.retrievers import BM25Retriever

# LangChain 1.0 moved several long-standing retrievers into a companion package
# called `langchain-classic`. Same class, same behaviour, new address. The
# try/except keeps this notebook working on either version rather than pinning
# you to one -- a pattern worth knowing, because library reshuffles like this
# are routine and the fix is almost always "the import moved".
try:
    from langchain_classic.retrievers import EnsembleRetriever      # LangChain 1.x
except ImportError:
    from langchain.retrievers import EnsembleRetriever              # LangChain 0.3

def build_retrievers(namespace, chunks, k=5):
    vector = STORES[namespace].as_retriever(search_kwargs={'k': k})
    keyword = BM25Retriever.from_documents(chunks); keyword.k = k
    hybrid = EnsembleRetriever(retrievers=[vector, keyword], weights=[0.5, 0.5])
    return {'vector': vector, 'keyword': keyword, 'hybrid': hybrid}

R = {'fixed':    build_retrievers('fixed', FIXED),
     'semantic': build_retrievers('semantic', SEMANTIC)}
print('retrievers ready:', list(R['semantic']))

retrievers ready: ['vector', 'keyword', 'hybrid']


In [23]:
from src.to_documents import citation

Q = "What was AMD's Data Center segment revenue?"
print('QUESTION:', Q, '\n')
for mode in ('vector', 'keyword', 'hybrid'):
    print(f'--- {mode.upper()}')
    for d in R['semantic'][mode].invoke(Q)[:3]:
        print(f'   {citation(d)}')
        print(f'      {d.page_content[:95].strip()}...')
    print()

QUESTION: What was AMD's Data Center segment revenue? 

--- VECTOR
   [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
      ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS...
   [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
      Additional information on our reportable segments is contained in Note 4 - Segment Reporting of...
   [AMD · 2026-02-04 · Item 1 - Business]
      and our consolidated subsidiaries. AMD drives innovation in high performance and AI computing t...

--- KEYWORD
   [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
      Additional information on our reportable segments is contained in Note 4 - Segment Reporting of...
   [AMD · 2026-02-04 · Item 8 - Financial Statements and Supplementary Data]
      Before segment change | After segment change
(in millions) | Data Center | Embedded | Client |...
   [NVIDIA · 2026-02-25 · Item 6 - Selected Financial Data]
      Compute & Networking revenue - The year ov

## 9 · Comparison #1 — fixed vs semantic

Same questions, same retrieval, same everything — only the chunking differs.
Read the passages and judge which set actually contains the answer.

This is a **qualitative** comparison on purpose. Session 2 was explicit that
formal retrieval metrics are Week 4 material. Looking at what came back and
saying which is better is the right level here.

In [24]:
COMPARE_QS = [
    "What was AMD's Data Center segment revenue?",
    "What reasons does NVIDIA give for the change in its revenue?",
    "What export control risks are disclosed?",
    "What were Intel's total operating expenses?",
]

for q in COMPARE_QS:
    print('='*78); print('Q:', q)
    for name in ('fixed', 'semantic'):
        print(f'\n  [{name}]')
        for d in R[name]['hybrid'].invoke(q)[:3]:
            tag = 'TABLE' if d.metadata['is_table'] else 'text '
            print(f'    {tag} {citation(d)}')
            print(f'          {d.page_content[:110].strip()}...')
    print()

Q: What was AMD's Data Center segment revenue?

  [fixed]
    text  [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
          In 2025, we delivered strong annual revenue growth with net revenue increasing 34% to $34.6 billion, compared...
    text  [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
          Data Center net revenue of $16.6 billion in 2025 increased by 32%, compared to net revenue of $12.6 billion in...
    TABLE [AMD · 2026-02-04 · Item 8 - Financial Statements and Supplementary Data]
          Before segment change | After segment change
(in millions) | Data Center | Embedded | Client | Gaming | Client...

  [semantic]
    text  [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
          Additional information on our reportable segments is contained in Note 4 - Segment Reporting of the Notes to F...
    text  [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
          ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF 

**Write down what you notice.** For each question: did one strategy return
the passage that actually holds the answer while the other missed it? Table
lookups are usually clearest — watch whether the table arrives whole with its
header attached to its numbers.

A paragraph plus two examples fully answers this requirement.

## 10 · Comparison #2 — reranking

Session 1 named this as the Week 2 experiment: build it without reranking,
build it with, compare.

Why it helps: retrieval compares two sets of numbers that were calculated
separately — the passage never actually *sees* your question. A **cross-encoder**
reads the question and one passage together and scores the pair. Far more
accurate, far too slow to run over thousands of passages.

So each does what it's good at. Retrieval casts a wide cheap net (20
candidates); the reranker makes a careful decision about those 20 and keeps 5.

**First run downloads the model** — a few hundred megabytes. It runs on your
Mac's processor rather than a graphics card, so it's slower than a cloud
service would be. That's expected, not broken.

In [27]:
try:
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
except ImportError:
    from langchain.retrievers import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

cross_encoder = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)

wide = build_retrievers('semantic', SEMANTIC, k=20)['hybrid']
RERANKED = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=wide)
PLAIN    = R['semantic']['hybrid']
print('reranker ready')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9789.42it/s]


reranker ready


In [28]:
for q in ["What was AMD's Data Center segment revenue?",
          "What risks does NVIDIA disclose about competition?"]:
    print('='*78); print('Q:', q, '\n')
    for label, ret in (('WITHOUT rerank', PLAIN), ('WITH rerank', RERANKED)):
        t0 = time.time()
        docs = ret.invoke(q)[:3]
        print(f'  {label}  ({time.time()-t0:.2f}s)')
        for d in docs:
            print(f'     {citation(d)}')
            print(f'        {d.page_content[:88].strip()}...')
        print()

Q: What was AMD's Data Center segment revenue? 

  WITHOUT rerank  (1.25s)
     [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
        Additional information on our reportable segments is contained in Note 4 - Segment Repor...
     [AMD · 2026-02-04 · Item 6 - Selected Financial Data]
        ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERA...
     [AMD · 2026-02-04 · Item 8 - Financial Statements and Supplementary Data]
        Before segment change | After segment change
(in millions) | Data Center | Embedded | Cl...

  WITH rerank  (1.78s)
     [AMD · 2026-02-04 · Item 8 - Financial Statements and Supplementary Data]
        The financial results of these acquired businesses, which were not material, were includ...
     [Intel · 2026-01-23 · Item 8 - Financial Statements and Supplementary Data]
        (In Millions) | Dec 28, 2024 | Divestitures | Transfers | Impairments | Dec 27, 2025
Cli...
     [AMD · 2026-02-04 · Item 8 - Financial 

**Record both columns.** Did the top passage change? Did something more
relevant move up? And how much slower was it?

Reporting the cost matters as much as the benefit. A reranker that buys better
passages for two extra seconds is a trade-off, not a free win — saying so is
exactly the kind of judgement the write-up is looking for.

In [46]:
import time
from statistics import mean

QS     = [r['question'] for r in TEST_QUESTIONS]
ALL_CO = sorted({c.metadata['company'] for c in SEMANTIC if c.metadata['company']})

def named(q):
    """Which company the question names, if exactly one."""
    hits = [c for c in ALL_CO if c.lower() in q.lower()]
    return hits[0] if len(hits) == 1 else None

def profile(retriever, k=5):
    """Retrieve for every question and record what came back."""
    out = []
    for q in QS:
        t0   = time.perf_counter()
        docs = retriever.invoke(q)[:k]
        dt   = time.perf_counter() - t0
        want = named(q)
        out.append({
            'q':      q,
            'want':   want,
            'on':     sum(d.metadata['company'] == want for d in docs) if want else None,
            'tables': sum(bool(d.metadata['is_table']) for d in docs),
            'secs':   dt,
            'top1':   docs[0].page_content[:70] if docs else '',
        })
    return out

before = profile(PLAIN)
after  = profile(RERANKED)

print('| Question | on-company | tables | top-1 moved | seconds |')
print('|---|---|---|---|---|')
for b, a in zip(before, after):
    oc = f"{b['on']}/5 → {a['on']}/5" if b['want'] else '— not company-specific'
    print(f"| {b['q'][:42]}… | {oc} | {b['tables']} → {a['tables']} | "
          f"{'yes' if b['top1'] != a['top1'] else 'no'} | "
          f"{b['secs']:.2f} → {a['secs']:.2f} |")

pairs = [(b, a) for b, a in zip(before, after) if b['want']]
print(f"\non-company (4 company-specific questions):"
      f"  no rerank {mean(b['on'] for b,_ in pairs):.2f}/5"
      f"  →  reranked {mean(a['on'] for _,a in pairs):.2f}/5")
print(f"tables retrieved (all 8):"
      f"  no rerank {mean(b['tables'] for b in before):.2f}/5"
      f"  →  reranked {mean(a['tables'] for a in after):.2f}/5")
print(f"median latency:"
      f"  no rerank {sorted(b['secs'] for b in before)[4]:.2f}s"
      f"  →  reranked {sorted(a['secs'] for a in after)[4]:.2f}s")

| Question | on-company | tables | top-1 moved | seconds |
|---|---|---|---|---|
| What was NVIDIA's total revenue in the mos… | 5/5 → 4/5 | 1 → 1 | yes | 2.06 → 2.85 |
| What was AMD's Data Center segment revenue… | 4/5 → 4/5 | 1 → 3 | yes | 1.15 → 1.89 |
| Compare R&D spending across these companie… | — not company-specific | 3 → 1 | yes | 0.48 → 1.52 |
| What supply chain risks does AMD disclose?… | 3/5 → 3/5 | 0 → 0 | yes | 4.77 → 2.18 |
| What export control risks are disclosed ac… | — not company-specific | 0 → 0 | yes | 0.49 → 1.62 |
| How is the company performing?… | — not company-specific | 0 → 0 | yes | 0.31 → 1.62 |
| What is NVIDIA's forecast revenue for fisc… | 4/5 → 3/5 | 1 → 2 | yes | 0.36 → 1.40 |
| What cash bonus was paid to the CEO in Mar… | — not company-specific | 1 → 2 | yes | 4.05 → 1.71 |

on-company (4 company-specific questions):  no rerank 4.00/5  →  reranked 3.50/5
tables retrieved (all 8):  no rerank 0.88/5  →  reranked 1.12/5
median latency:  no rerank 1.

In [47]:
class Filtered:
    """Route each question to its own company's retriever; fall back to all."""
    def invoke(self, q):
        return BY_COMPANY[named(q) or ALL].invoke(q)

filtered = profile(Filtered())
pairs_f  = [(b, f) for b, f in zip(before, filtered) if b['want']]

print(f"on-company:  hybrid {mean(b['on'] for b,_ in pairs_f):.2f}/5"
      f"  →  filtered+rerank {mean(f['on'] for _,f in pairs_f):.2f}/5")
print(f"tables:      hybrid {mean(b['tables'] for b in before):.2f}/5"
      f"  →  filtered+rerank {mean(f['tables'] for f in filtered):.2f}/5")
print(f"latency:     hybrid {sorted(b['secs'] for b in before)[4]:.2f}s"
      f"  →  filtered+rerank {sorted(f['secs'] for f in filtered)[4]:.2f}s")

on-company:  hybrid 4.00/5  →  filtered+rerank 5.00/5
tables:      hybrid 0.88/5  →  filtered+rerank 0.88/5
latency:     hybrid 1.15s  →  filtered+rerank 2.74s


## 11 · Answers — cited, or refused

The prompt gives the model three exits and requires it to take one:

- **ANSWER** — the passages contain it → answer, citing passage numbers
- **CLARIFY** — the question doesn't say which company or year → ask
- **REFUSE** — the passages don't contain it → say so

The third matters most. Session 2's four-quadrant picture had one genuinely
dangerous box: retrieval finds nothing and the model answers confidently
anyway. For an analyst, a confident wrong number is worse than no answer.

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from src.prompts import SYSTEM, USER

prompt = ChatPromptTemplate.from_messages([('system', SYSTEM), ('human', USER)])

def format_docs(docs):
    return '\n\n'.join(
        f"[{i}] {d.metadata.get('company','')} · {d.metadata.get('filing_date','')} "
        f"· {d.metadata.get('section','')}\n{d.page_content[:1800]}"
        for i, d in enumerate(docs, 1))

def ask(question, retriever=None, show_sources=True):
    retriever = retriever or RERANKED
    docs = retriever.invoke(question)[:5]
    answer = llm.invoke(prompt.invoke(
        {'context': format_docs(docs), 'question': question})).content.strip()
    if show_sources and docs:
        seen = []
        for d in docs:
            c = citation(d)
            if c not in seen: seen.append(c)
        answer += '\n\nSources considered:\n' + '\n'.join('  ' + s for s in seen)
    return answer, docs

print('ready')

ready


In [30]:
for q in ["What was NVIDIA's total revenue in the most recent fiscal year?",  # answers
          "How is the company performing?",                                   # asks which
          "What is NVIDIA's forecast revenue for fiscal year 2028?"]:          # refuses
    a, _ = ask(q)
    print('='*78); print('Q:', q); print(); print(a); print()

Q: What was NVIDIA's total revenue in the most recent fiscal year?

REFUSE — the passages do not contain the answer. The provided excerpts do not include NVIDIA's total revenue for the most recent fiscal year. While some revenue drivers and segment performance are discussed, the consolidated statement of income or a specific revenue figure is not present in the retrieved passages.

Sources considered:
  [NVIDIA · 2026-02-25 · Item 15 - Exhibits and Financial Statement Schedules]
  [NVIDIA · 2026-02-25 · Item 4 - Mine Safety Disclosures]
  [Intel · 2026-01-23 · Item 7 - Management's Discussion and Analysis]
  [NVIDIA · 2026-02-25 · Item 6 - Selected Financial Data]

Q: How is the company performing?

CLARIFY — Which company and fiscal period are you asking about?

Sources considered:
  [Texas Instruments · 2026-02-06 · Item 3 - Legal Proceedings]
  [AMD · 2026-02-04 · Item 1A - Risk Factors]
  [AMD · 2026-02-04 · Item 8 - Financial Statements and Supplementary Data]
  [Intel · 2026-01-2

> The third question is the important one. Forward-looking guidance isn't in a
> 10-K — it lives in earnings releases. If your app answers it with a number,
> that's a **generation** problem to fix in the prompt, not a retrieval one.

In [42]:
import importlib, src.prompts
importlib.reload(src.prompts)
from src.prompts import SYSTEM, USER

prompt = ChatPromptTemplate.from_messages([('system', SYSTEM), ('human', USER)])
print('prompt reloaded —', len(SYSTEM), 'characters')

prompt reloaded — 2678 characters


## 12 · The evaluation table

Evaluation is **bonus credit** this week — Week 4 covers it properly, and the
sessions said plainly not to build a metrics framework now.

What they asked for is a short table: realistic questions, and for each — did
the right passage come back, is the answer grounded and cited, what happened.
Plus **at least one failure**, explained as either a retrieval problem or a
generation problem. Those have completely different fixes, and telling them
apart is the entire point.

Run the questions, read the passages, then make one judgement call per row.

In [43]:
import re

TEST_QUESTIONS = [
    {'question': "What was NVIDIA's total revenue in the most recent fiscal year?"},
    {'question': "What was AMD's Data Center segment revenue?"},
    {'question': "Compare R&D spending across these companies."},
    {'question': "What supply chain risks does AMD disclose?"},
    {'question': "What export control risks are disclosed across these filings?"},
    {'question': "How is the company performing?", 'should_ask': True},
    {'question': "What is NVIDIA's forecast revenue for fiscal year 2028?", 'should_refuse': True},
    {'question': "What cash bonus was paid to the CEO in March 2026?", 'should_refuse': True},
]

for r in TEST_QUESTIONS:
    answer, docs = ask(r['question'], show_sources=False)
    r['answer'] = answer
    r['n_citations'] = len(set(re.findall(r'\[(\d{1,2})\]', answer)))
    print('='*78); print('Q:', r['question'])
    print('\nRETRIEVED:')
    for d in docs[:3]:
        print(f'   {citation(d)}')
        print(f'      {d.page_content[:80].strip()}...')
    print('\nANSWER:', answer[:400]); print()

Q: What was NVIDIA's total revenue in the most recent fiscal year?

RETRIEVED:
   [NVIDIA · 2026-02-25 · Item 15 - Exhibits and Financial Statement Schedules]
      We have audited the accompanying consolidated balance sheets of NVIDIA Corporati...
   [NVIDIA · 2026-02-25 · Item 4 - Mine Safety Disclosures]
      1/31/2021 | 1/30/2022 | 1/29/2023 | 1/28/2024 | 1/26/2025 | 1/25/2026
NVIDIA Cor...
   [NVIDIA · 2026-02-25 · Item 15 - Exhibits and Financial Statement Schedules]
      (1) Represents reclassifications from non-marketable equity securities to market...

ANSWER: REFUSE — the passages retrieved do not contain NVIDIA's total revenue for the most recent fiscal year. While several financial figures are provided, such as accounts receivable concentrations, inventory balances, and property and equipment values, the total revenue figure is not stated in any of the provided passages.

Q: What was AMD's Data Center segment revenue?

RETRIEVED:
   [AMD · 2026-02-04 · Item 8 - Financial 

In [44]:
r = [x for x in TEST_QUESTIONS if 'supply chain' in x['question']][0]
print(r['answer'])
print('\ncitations found:', r['n_citations'])

AMD discloses that IT outages, data loss, data breaches, and cyberattacks could disrupt operations and compromise intellectual property or other sensitive information, potentially causing significant damage to its business, reputation, financial condition, and results of operations. The company relies on technology hardware, software, cloud services, infrastructure, networks, and systems (collectively, IT Systems), some of which are managed internally and others provided by third parties. AMD processes and maintains sensitive data, including personal information and intellectual property, and faces increasing cybersecurity threats ranging from individual hackers to state-sponsored attackers. These threats may be generic or specifically targeted against AMD’s IT Systems or supply chain, and the prevalence of remote working arrangements adds additional operational risks and attack vectors [5].

citations found: 1


In [35]:
_, docs = ask("What supply chain risks does AMD disclose?", show_sources=False)
for i, d in enumerate(docs, 1):
    print(i, d.metadata['company'], '·', d.metadata['section'])
    print('   ', d.page_content[:180].replace('\n', ' '), '\n')

1 AMD · Item 8 - Financial Statements and Supplementary Data
    References herein to AMD or the Company mean Advanced Micro Devices, Inc. and its consolidated subsidiaries. AMD's products include Artificial Intelligence (AI) Accelerators, micro 

2 AMD · Item 1A - Risk Factors
    From time to time, suppliers may extend lead times, limit supply or increase prices due to capacity constraints or other factors. Also, some of these materials and components may b 

3 AMD · Item 1 - Business
    While compliance has not historically had a material impact on our financial condition, earnings, or competitive position, there can be no assurance that these evolving laws will n 

4 Intel · Item 1A - Risk Factors
    to continue to rely upon third-party foundries. Delays in the development of foundries' future manufacturing processes could delay the introduction of products or components we des 

5 NVIDIA · Item 1A - Risk Factors
    These subcontractors assist with procuring components used in o

In [45]:
# Your judgement call. Read the RETRIEVED passages printed above and fill in
# 'yes', 'partly' or 'no' for each. This is the one human step, and it is
# the actual assignment — the code just produced something to judge.
VERDICTS = {
    "What was NVIDIA's total revenue in the most recent fiscal year?": 'no',
    "What was AMD's Data Center segment revenue?":                     'yes',
    "Compare R&D spending across these companies.":                    'partly',
    "What supply chain risks does AMD disclose?":                      'no',
    "What export control risks are disclosed across these filings?":   'yes',
}
for r in TEST_QUESTIONS:
    r['retrieved_ok'] = VERDICTS.get(r['question'], '')
print('verdicts recorded')

verdicts recorded


In [37]:
from src.scorecard import scorecard, failures

print(scorecard(TEST_QUESTIONS))
print()
print(failures(TEST_QUESTIONS))

| Test question | Right evidence retrieved? | Answer grounded and cited? | What happened |
|---|---|---|---|
| What was NVIDIA's total revenue in the most recent fiscal yea… | No | n/a — declined | correct — no evidence, and it said so |
| What was AMD's Data Center segment revenue? | Yes | Yes | correct |
| Compare R&D spending across these companies. | Partly | n/a — declined | RETRIEVAL — only some of the evidence came back |
| What supply chain risks does AMD disclose? | Yes | Yes | correct |
| What export control risks are disclosed across these filings? | Yes | Yes | correct |
| How is the company performing? | n/a — question underspecified | Yes | correct — declined instead of guessing |
| What is NVIDIA's forecast revenue for fiscal year 2028? | No evidence exists | Yes | correct — declined instead of guessing |
| What cash bonus was paid to the CEO in March 2026? | No evidence exists | Yes | correct — declined instead of guessing |

RETRIEVAL FAULTS (1)
Fix the retrieval side:

**That table is your evaluation deliverable.** Paste it into the write-up.

Then pick one failure and write two or three sentences: what went wrong, was
it retrieval or generation, and what you changed or would change. That single
paragraph is what the sessions said makes this section strong.

## 13 · The chat interface

The bonus deliverable, and what you record for the video.

It has a **company dropdown** as well as the message box. Leaving it on *All
companies* searches everything — which is what cross-company questions need.
Picking one narrows the search to that company's passages before it runs.

That narrowing is **metadata filtering**, the technique demoed in
Session 1 with its `doc_type` and `product_area` filters. Your chunks already
carry a `company` tag, so it's nearly free — and it's worth calling out in the
video as a named technique from the course.

The next cell starts a small web server on your Mac and prints a `localhost`
address. `share=True` also gives a public link good for 72 hours. Press the
cell's **stop button** when you're finished.

In [53]:
import gradio as gr

PASTELS = """
.gradio-container, .gradio-container .main { background: #e7f4ea !important; }

#finrag-chat {
    background: #fbfffc !important;
    border: 2px solid #a8d5b5 !important;
    border-radius: 16px !important;
}
#finrag-chat [data-testid="user"], #finrag-chat .user { background: #cfe8d5 !important; }
#finrag-chat [data-testid="bot"],  #finrag-chat .bot  { background: #fdf3e0 !important; }

#finrag-picker, #finrag-picker input, #finrag-picker .wrap {
    background: #d8e9f5 !important;
    border-radius: 12px !important;
}
#finrag-picker label span { color: #2b5566 !important; font-weight: 600 !important; }

[class*="example"] button, button[class*="example"], .examples button {
    background: #f6e2ef !important;
    border: 1px solid #e0b8d2 !important;
    border-radius: 12px !important;
    color: #5a3350 !important;
}
[class*="example"] button:hover, .examples button:hover { background: #efd0e5 !important; }

textarea, input[type="text"] { background: #ffffff !important; border-radius: 12px !important; }
footer { display: none !important; }
"""

def chat_fn(message, history, company):
    t0 = time.time()
    answer, docs = ask(message, retriever=BY_COMPANY[company], show_sources=True)
    scope = 'all companies' if company == ALL else company
    return answer + f"\n\n*{time.time()-t0:.1f}s · {len(docs)} passages · searched {scope}*"

with gr.Blocks(theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='teal'),
               css=PASTELS, title='FinRAG') as demo:

    gr.Markdown('# FinRAG — semiconductor 10-K intelligence')
    gr.Markdown('Ask about eight semiconductor companies from their latest annual '
                'filings. Every answer is cited. Questions the filings cannot '
                'answer are refused, not guessed.')

    # Rendered here, so it appears exactly here rather than in ChatInterface's
    # collapsed "Additional Inputs" accordion. Passing the same object below
    # wires it up without drawing it twice.
    picker = gr.Dropdown(choices=[ALL] + COMPANIES, value=ALL,
                         label='Search within', elem_id='finrag-picker')

    gr.ChatInterface(
        chat_fn,
        chatbot=gr.Chatbot(elem_id='finrag-chat', height=460),
        additional_inputs=[picker],
        examples=[["What was NVIDIA's Data Center revenue in the most recent fiscal year?", ALL],
                  ['Which of these companies reported the highest revenue?', ALL],
                  ['What export control risks do these filings disclose?', ALL],
                  ['How is the company performing?', ALL],
                  ["What is NVIDIA's forecast revenue for fiscal 2028?", ALL]])

demo.launch(share=True)

/var/folders/zc/gzc08x0938z8ftdp_s5pkjbr0000gn/T/ipykernel_76764/1806405497.py:38: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='teal'),


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://f5d54d32bb0f1eed7d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [50]:
demo.close()

Closing server running on port: 7860


---
## Done

A working cited RAG pipeline with a refusal path, both required comparisons,
an evaluation table with a diagnosed failure, and a live interface.

Commit and push, then write up and record.